# Convolutional Neural Networks: Step by Step

We will be implementing the building blocks of a CNN!

- Convolution functions, including:
    - Zero Padding
    - Convolve window
    - Convolution forward
    - Convolution backward (optional)
- Pooling functions, including:
    - Pooling forward
    - Create mask
    - Distribute value
    - Pooling backward (optional)

This notebook will implement these functions from scratch in `numpy`. In the next notebook, you will use the TensorFlow equivalents of these functions to build the simple model:

<img src="images/model.png" style="width:800px;height:300px;">

## Import Library

In [13]:
import numpy as np
import matplotlib.pyplot as plt

## 1. CNN

### 1.1 - Zero-Padding

Zero-padding adds 0 around the border of an image:
<img src="images/PAD.png" style="width:600px;height:400px;">
<caption><center> <u> <font color='purple'> <b>Figure 1</b> </u><font color='purple'>  : <b>Zero-Padding</b><br> Image (3 channels, RGB) with a padding of 2. </center></caption>

The main benefits of padding are:

- It **allows** you to use a CONV layer **without necessarily shrinking the height and width of the volumes**. This is important for building deeper networks.

- It helps us **keep more of the information at the border** of an image. Without padding, very few values at the next layer would be affected by pixels at the edges of an image.

In [6]:
def zero_pad(X, pad):
    """
    Argument:
    X -- np.array of shape (m, n_H, n_W, n_C)

    Return:
    X_pad -- np.array of shape (m, n_H + 2 * pad, n_W + 2 * pad, n_C)
    """
    X_pad = np.pad(X, ((0, 0), (pad, pad), (pad, pad), (0, 0)), mode='constant', constant_values= (0, 0))
    return X_pad

### 1.2 - Single Step of Convolution

In this part, implement a single step of convolution, in which you apply the filter to a single position of the input. This will be used to build a convolutional unit, which:

- Takes an input volume
- Applies a filter at every position of the input
- Outputs another volume (usually of different size)

<img src="images/Convolution_schematic.gif" style="width:500px;height:300px;">
<caption><center> <u> <font color='purple'> <b>Figure 2</b> </u><font color='purple'>  : <b>Convolution operation</b><br> with a filter of 3x3 and a stride of 1 (stride = amount you move the window each time you slide) </center></caption>

In [7]:
def conv_single_step(a_slice_input_prev, W, b):
    """
    Apply one filter defined by parameters W on a single slice (a_slice_prev) of the output activation
    of the previous layer.

    Argument:
    a_slice_input_prev -- a slice of input data of shape (f, f, n_C_prev)
    W -- Weight parameters contained in a window - matrix of shape (f, f, n_C_prev)
    b -- Bias parameters contained in a window - matrix of shape (1, 1, 1)

    Return:
    Z -- a scalar value, the result of convolving the sliding window (W, b) on a slice x of the input data
    """
    s = a_slice_input_prev * W #element-wise
    Z = np.sum(s)
    Z += float(b)

    return Z

### 1.3 - CNN - Forward Pass

In the forward pass, you will take many filters and convolve them on the input. Each 'convolution' gives you a 2D matrix output. You will then stack these outputs to get a 3D volume:

<img src="images/conv_kiank.png" style="width:500px;height:300px;">
<caption><center> <u> <font color='purple'> <b>Figure 3</b> </u><font color='purple'>  : <b>Convolution on 3D volume</b><br> with a filter of fxf </center></caption>

**Reminder**:

The formulas relating the output shape of the convolution to the input shape are:

$$n_H = \Bigl\lfloor \frac{n_{H_{prev}} - f + 2 \times pad}{stride} \Bigr\rfloor +1$$
$$n_W = \Bigl\lfloor \frac{n_{W_{prev}} - f + 2 \times pad}{stride} \Bigr\rfloor +1$$
$$n_C = \text{number of filters used in the convolution}$$

To define a_slice you will need to first define its corners `vert_start`, `vert_end`, `horiz_start` and `horiz_end`. This figure may be helpful for you to find out how each of the corners can be defined using h, w, f and s in the code below.

<img src="images/vert_horiz_kiank.png" style="width:400px;height:300px;">
<caption><center> <u> <font color='purple'> <b>Figure 4</b> </u><font color='purple'>  : <b>Definition of a slice using vertical and horizontal start/end (with a 2x2 filter)</b> <br> This figure shows only a single channel.  </center></caption>

In [17]:
def conv_forward(A_prev, W, b, hparameters):
    """
    Implements the forward propagation for a convolution function

    Argument:
    A_prev -- output activations of the previous layer, numpy array of shape (m, n_H_prev, n_W_prev, n_C_prev)
    (Với layer đầu tiên (conv đầu): A_prev chính là input ảnh (X). Với các layer sau: A_prev là output (activation) của layer trước đó)

    W -- Weights/Filter window, numpy array of shape (f, f, n_C_prev, n_C)

    b -- Biases, numpy array of shape (1, 1, 1, n_C). Each filter has its own (single) bias.

    hparameters -- python dictionary containing "stride" and "pad"

    Returns:
    Z -- conv output, numpy array of shape (m, n_H, n_W, n_C)
    cache -- cache of values needed for the conv_backward() function
    """

    # 1. Get A_prev shape
    m, n_H_prev, n_W_prev, n_C_prev = A_prev.shape

    #2. Get W shape
    f, f, n_C_prev, n_C = W.shape

    #3. Get hparameters
    stride = hparameters["stride"]
    pad = hparameters["pad"]

    #4. Get CONV shape
    n_H = int((n_H_prev + 2 * pad - f)/stride) + 1
    n_W = int((n_W_prev + 2 * pad - f)/stride) + 1

    #5. Initialize output volume Z with zeros
    Z = np.zeros((m, n_H, n_W, n_C))

    #6. Create A_prev_pad by padding A_prev
    A_prev_pad = zero_pad(A_prev, pad)

    for i in range(m):
        a_prev_pad = A_prev_pad[i]
        for h in range(n_H):
            # Find the vertical start and end of the current "slice"
            vert_start = h * stride
            vert_end = vert_start + f

            for w in range(n_W):
                # Find the horizontal start and end of the current "slice"
                horiz_start = w * stride
                horiz_end = horiz_start + f

                for c in range(n_C):
                    # Use the corners to define the (3D) slice of a_prev_pad
                    a_slice_prev = a_prev_pad[vert_start:vert_end,horiz_start:horiz_end,:]

                    # Convolve the (3D) slice with the correct filter W and bias b, to get back one output neuron.
                    weights = W[:, :, :, c]
                    biases = b[:, :, :, c]
                    Z[i, h, w, c] = conv_single_step(a_slice_prev, weights, biases)

    # Save information in "cache" for the backprop
    cache = (A_prev, W, b, hparameters)

    return Z, cache